# 02 — Home Credit + BISG scale story (architecture)

**Phase:** P4 (R-10 §2) — *documented architecture only; no CI gate.*

German Credit can demonstrate `sex` bias layers but **cannot** support BISG: no US surname + geography fields. Every German Credit report §II limitations block states that explicitly.

This notebook documents how a Home Credit–scale race/ethnicity **proxy** path would attach to the workbench bias battery, and runs a **small illustrative** `surgeo` BISG panel so the wiring is concrete — without shipping a production module or Kaggle dump into the repo.

| Authority | Role |
| --------- | ---- |
| R-03 | BISG methodology + Home Credit as scale dataset |
| R-07 | `surgeo` stack choice + 2020 Census DHC caveat |
| R-08 / R-09 | State-law / scope context for proxy use |
| deep-research-v1 §10.2 V4/V5 | surgeo census lag; CFPB Stata port fallback |

**Not in scope for P4:** new `workbench` module, `phase4` pytest floor, CI label branch, or vendoring the full Home Credit 10k subset.


## §0 Setup


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
os.chdir(ROOT)
print("repo root:", ROOT)

SAMPLE = ROOT / "data" / "illustrative" / "home_credit_bisg_proxy_sample.csv"
assert SAMPLE.exists(), SAMPLE


## §1 Why German Credit stops at sex

| Need for BISG | German Credit | Home Credit Default Risk (Kaggle) |
| ------------- | ------------- | --------------------------------- |
| Surname | ❌ | Partial / not US-native (`NAME_*` fields vary by release; often absent in public tables) |
| US geography (ZCTA / tract) | ❌ | ❌ (non-US book) |
| Direct race/ethnicity | ❌ | ❌ |
| Direct sex | ✅ derived from `personal_status` | `CODE_GENDER` available |

**Implication:** a true US fair-lending race/ethnicity outcomes analysis needs either HMDA self-reported fields (mortgage) or a **BISG (or BIFSG) proxy** on surname + geocode. Home Credit is the *scale* story in the research plan; it still needs a geography strategy (US-population stand-in, or treat the notebook as architecture-only). P4 chooses the latter with an illustrative US panel.


## §2 Target architecture (not implemented as a package module)

```
Home Credit application_train (~10k subset)
        │
        ▼
column map / enrichment
  NAME_FAMILY  ──► surname
  region / ZIP ──► ZCTA (US stand-in or geocoder; see caveats)
  SCORE / decision columns for Layer A
        │
        ▼
surgeo.SurgeoModel.get_probabilities(names, zcta)
        │
        ▼
proxy probabilities: white, black, api, native, multiple, hispanic
        │
        ▼
bias_battery Layer C (proxy correlation + auxiliary classifier)
  — protected attrs NEVER enter the scoring model —
        │
        ▼
Appendix B + audit bias_battery.result (jurisdiction_frame)
```

**Workbench integration (future, out of P4):** a `protected_attributes[].source: bisg_proxy` path in config would point at a precomputed proxy table (or a thin adapter). P0–P3 only implement `source: direct` for `sex`.


## §3 Illustrative sample (synthetic US panel)

File: `data/illustrative/home_credit_bisg_proxy_sample.csv`

Column contract intended for a future Home Credit mapper:

| Column | Role |
| ------ | ---- |
| `NAME_FAMILY` | Surname input to BISG |
| `ZCTA` | 5-digit ZIP/ZCTA geography |
| `score` / `approval_binary` | Stand-ins for model outputs used in Layer A sketches |


In [ ]:
df = pd.read_csv(SAMPLE, dtype={"ZCTA": str})
df["ZCTA"] = df["ZCTA"].str.zfill(5)
df.head()


## §4 Run `surgeo` BISG (Bayesian Improved Surname Geocoding)

Uses Census **2010** surname + ZCTA tables bundled with `surgeo` 1.1.2 (see V4 caveat below).


In [ ]:
from surgeo import SurgeoModel

model = SurgeoModel(geo_level="ZCTA")
bisg = model.get_probabilities(df["NAME_FAMILY"], df["ZCTA"])
bisg.head(10)


## §5 Illustrative Layer C sketch (proxy ↔ score)

Not a full Fairlearn/Aequitas battery — just the correlation shape a production Layer C would start from.


In [ ]:
race_cols = ["white", "black", "api", "native", "multiple", "hispanic"]
joined = df.join(bisg[race_cols])

# Spearman correlation between model score and each BISG race probability
corr = joined[race_cols + ["score"]].corr(method="spearman")["score"].drop("score")
corr_table = corr.rename("spearman_vs_score").to_frame()
corr_table["abs"] = corr_table["spearman_vs_score"].abs()
corr_table = corr_table.sort_values("abs", ascending=False).drop(columns="abs")
corr_table

# Optional: approval rate by majority-proxy race label (argmax) — pedagogical only
joined["proxy_argmax"] = bisg[race_cols].idxmax(axis=1)
approval_by_proxy = (
    joined.groupby("proxy_argmax")["approval_binary"]
    .agg(n="size", approval_rate="mean")
    .sort_values("n", ascending=False)
)
approval_by_proxy


## §6 Methodology caveats (must stay in shipping copy)

1. **surgeo census vintage (V4):** PyPI `surgeo` 1.1.2 ships **2010** tables; CFPB methodology references a **2020 Census DHC** update (April 2024). Probabilities here may diverge from a 2020-calibrated examiner tool.
2. **Fallback (V5):** if divergence is material for a US production book, prefer a maintained Python port of CFPB `proxy-methodology` (Stata reference) over silent reliance on surgeo.
3. **Geocoder sensitivity:** CFPB uses ArcGIS; alternate geocoders → different ZCTA/tract assignments → different BISG probabilities (R-07).
4. **Home Credit is not a US lending population.** Using it for BISG without a documented US geography strategy would over-claim. This notebook uses a **synthetic US panel** to demonstrate wiring only.
5. **Proxies ≠ labels.** BISG outputs are probabilistic; never feed them into the credit model; use only in validation outcomes / proxy-risk layers.
6. **No compliance claim.** Outputs are review-support architecture evidence, not an ECOA determination.

## §7 Exit criterion checklist (P4)

- [x] `notebooks/02_home_credit_bias_proxy.ipynb` documents Home Credit + BISG architecture
- [x] README section points here
- [x] Runnable `surgeo` cell on illustrative sample
- [x] No CI `phase:p4` floor (intentional — R-10 exit: documented architecture only)
- [ ] Optional follow-on: 10k Home Credit subset + geography map + optional thin `bisg_proxy` config source
